In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
nibrs_panel = pd.read_csv("nibrs/nibrs_panel.csv", index_col = 0)
ccc_panel = pd.read_csv("ccc/ccc_panel.csv", index_col = 0)
crosswalk_panel = pd.read_csv("crosswalk/crosswalk_panel.csv", index_col = 0, dtype={
        'county_fips': str,
        'place_fips': str,
    })

In [3]:
nibrs_panel.head()

,ori,state,incident_date,ucr_offense_code
11,AL0010000,alabama,2017-12-11,arson
12,AL0010000,alabama,2017-11-01,destruction/damage/vandalism of property
14,AL0010100,alabama,2017-05-19,burglary/breaking and entering
15,AL0010100,alabama,2017-08-30,"stolen property offenses (receiving, selling, ..."
17,AL0010100,alabama,2017-12-27,"stolen property offenses (receiving, selling, ..."


In [4]:
crosswalk_panel.head()

,ori,county_fips,place_fips,state_name,county_name,place_name
0,AL0040200,01001,03220,ALABAMA,AUTAUGA,AUTAUGAVILLE TOWN
1,AL0040100,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY
2,AL0040300,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY
3,AL0040000,01001,99001,ALABAMA,AUTAUGA,AUTAUGA COUNTY
4,AL0051100,01003,04660,ALABAMA,BALDWIN,State of Alabama


## Merging the Crosswalk panel and the NIBRS panel on the ORI column

In [5]:
# 1. Make sure ORIs are clean strings in both dataframes
nibrs_panel['ori'] = nibrs_panel['ori'].astype(str).str.strip()
crosswalk_panel['ori'] = crosswalk_panel['ori'].astype(str).str.strip()

In [6]:
# 2. Merge crosswalk into NIBRS (left join: keep all NIBRS rows)
merged = nibrs_panel.merge(
    crosswalk_panel,
    on='ori',
    how='left'
)

In [7]:
merged

,ori,state,incident_date,ucr_offense_code,county_fips,place_fips,state_name,county_name,place_name
0,AL0010000,alabama,2017-12-11,arson,01073,99073,ALABAMA,JEFFERSON,JEFFERSON COUNTY
1,AL0010000,alabama,2017-11-01,destruction/damage/vandalism of property,01073,99073,ALABAMA,JEFFERSON,JEFFERSON COUNTY
2,AL0010100,alabama,2017-05-19,burglary/breaking and entering,01073,05980,ALABAMA,JEFFERSON,BESSEMER CITY
3,AL0010100,alabama,2017-08-30,"stolen property offenses (receiving, selling, ...",01073,05980,ALABAMA,JEFFERSON,BESSEMER CITY
4,AL0010100,alabama,2017-12-27,"stolen property offenses (receiving, selling, ...",01073,05980,ALABAMA,JEFFERSON,BESSEMER CITY
...,...,...,...,...,...,...,...,...,...
26702093,WVWSP6200,west virginia,2024-11-27,larceny/theft offenses - shoplifting,54039,14600,WEST VIRGINIA,KANAWHA,State of West Virginia
26702094,WVWSP6200,west virginia,2024-02-12,larceny/theft offenses - all other larceny,54039,14600,WEST VIRGINIA,KANAWHA,State of West Virginia
26702095,WVWSP6600,west virginia,2024-07-24,larceny/theft offenses - theft of motor vehicl...,54029,58372,WEST VIRGINIA,HANCOCK,State of West Virginia
26702096,WVWSP6600,west virginia,2024-06-14,larceny/theft offenses - theft of motor vehicl...,54029,58372,WEST VIRGINIA,HANCOCK,State of West Virginia


In [8]:
merged.isna().sum()

ori                     0
state                   0
incident_date           0
ucr_offense_code        0
county_fips         59477
place_fips          59477
state_name          59477
county_name         59477
place_name          59477
dtype: int64

In [9]:
missing_mask = (
    merged['county_fips'].isna() &
    merged['place_fips'].isna()  &
    merged['state_name'].isna()  &
    merged['county_name'].isna() &
    merged['place_name'].isna()
)

rows_with_all_missing = merged[missing_mask]
rows_with_all_missing

,ori,state,incident_date,ucr_offense_code,county_fips,place_fips,state_name,county_name,place_name
111202,AR0661700,arkansas,2017-10-30,burglary/breaking and entering,NaN,NaN,NaN,NaN,NaN
111203,AR0661700,arkansas,2017-01-31,larceny/theft offenses - theft from motor vehicle,NaN,NaN,NaN,NaN,NaN
111204,AR0661700,arkansas,2017-02-01,larceny/theft offenses - all other larceny,NaN,NaN,NaN,NaN,NaN
111205,AR0661700,arkansas,2017-05-05,larceny/theft offenses - all other larceny,NaN,NaN,NaN,NaN,NaN
210260,CO0191000,colorado,2017-02-02,larceny/theft offenses - theft from motor vehicle,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
26348091,VTVSP1600,vermont,2024-06-07,destruction/damage/vandalism of property,NaN,NaN,NaN,NaN,NaN
26348092,VTVSP1600,vermont,2024-08-03,burglary/breaking and entering,NaN,NaN,NaN,NaN,NaN
26348093,VTVSP1600,vermont,2024-09-01,larceny/theft offenses - theft from motor vehicle,NaN,NaN,NaN,NaN,NaN
26348094,VTVSP1600,vermont,2024-01-28,destruction/damage/vandalism of property,NaN,NaN,NaN,NaN,NaN


In [10]:
rows_with_all_missing.ori.unique()

array(['AR0661700', 'CO0191000', 'CO034019E', 'CT0007200', 'DE0011900',
       'DE001300X', 'DE002159E', 'DE0029Z0X', 'DE003350X', 'DE003500X',
       'IA0160700', 'IA0250700', 'IN0461500', 'KS0081300', 'KS0191200',
       'KS0490200', 'KS0780900', 'KS1051200', 'KY034019P', 'KY037015A',
       'KY037035Y', 'KY0400800', 'MA008049E', 'MA008249E', 'MA008269E',
       'MA009589E', 'MA009639E', 'MA009649E', 'MA011319E', 'MA013309E',
       'MA013390X', 'MA014640X', 'MA014649E', 'MA014669E', 'MA014709E',
       'MI2565000', 'MI631800X', 'MI631810X', 'MI631820X', 'MI631830X',
       'MI631840X', 'MI631850X', 'MI631860X', 'MI631870X', 'MI631880X',
       'MI631890X', 'MI631900X', 'MI631910X', 'MI631920X', 'MI8297400',
       'ND0270300', 'ND0510600', 'NH001010X', 'NH002010X', 'NH003010X',
       'NH004010X', 'NH005010X', 'NH006010X', 'NH007010X', 'NH008010X',
       'NH009010X', 'NH00927TZ', 'NH010010X', 'OH008130X', 'OH018A10X',
       'OH025A10X', 'OH060139E', 'OK0141100', 'OKOHP1200', 'ORDI

## Since the LEAIC was last released in 2012, it does not contain the mapping for these 99 jurisdictions. Other available crosswalks do not contain ORIs

## Hence we will drop these rows

In [11]:
merged = merged[merged['county_fips'].notna()].copy()

In [12]:
merged.isnull().sum()

ori                 0
state               0
incident_date       0
ucr_offense_code    0
county_fips         0
place_fips          0
state_name          0
county_name         0
place_name          0
dtype: int64

In [13]:
merged.shape

(26642621, 9)

##  Creating the County day and the place day panels

# =====================================================================
# 1. COUNTY–DAY PANEL (with zero-crime days)
# =====================================================================

In [14]:
# basic cleaning
merged['incident_date'] = pd.to_datetime(merged['incident_date'], errors='coerce')
merged = merged.dropna(subset=['incident_date'])
merged['county_fips'] = merged['county_fips'].astype(str).str.zfill(5)
merged['place_fips']  = merged['place_fips'].astype(str).str.zfill(5)

In [15]:
min_date = merged['incident_date'].min()
max_date = merged['incident_date'].max()
all_dates = pd.date_range(min_date, max_date, freq='D')

In [16]:
# Count incidents per county–day
county_counts = (
    merged
    .groupby(['county_fips', 'incident_date'], as_index=False)
    .size()
    .rename(columns={'size': 'property_crime_count'})
    .rename(columns={'incident_date': 'date'})
)

In [18]:
all_counties = county_counts['county_fips'].unique()

In [19]:
# Full grid: every county × every date
full_index = pd.MultiIndex.from_product(
    [all_counties, all_dates],
    names=['county_fips', 'date']
)

In [20]:
county_day = (
    county_counts
    .set_index(['county_fips', 'date'])
    .reindex(full_index)
    .reset_index()
)

In [21]:
# Fill missing counts with 0
county_day['property_crime_count'] = (
    county_day['property_crime_count'].fillna(0).astype(int)
)

In [22]:
county_day

,county_fips,date,property_crime_count
0,01003,2017-01-01,0
1,01003,2017-01-02,0
2,01003,2017-01-03,0
3,01003,2017-01-04,0
4,01003,2017-01-05,0
...,...,...,...
5414461,55139,2024-12-27,1
5414462,55139,2024-12-28,0
5414463,55139,2024-12-29,1
5414464,55139,2024-12-30,2


# =====================================================================
# 2. PLACE–DAY PANEL (with zero-crime days)
# =====================================================================

In [23]:
# Drop incidents with missing place_fips
merged_place = merged.dropna(subset=['place_fips']).copy()

In [24]:
# Count incidents per place–day
place_counts = (
    merged_place
    .groupby(['county_fips', 'place_fips', 'incident_date'], as_index=False)
    .size()
    .rename(columns={'size': 'property_crime_count'})
    .rename(columns={'incident_date': 'date'})
)

In [25]:
pairs = place_counts[['county_fips', 'place_fips']].drop_duplicates()
pairs['key'] = 1

In [26]:
# All dates
dates_df = pd.DataFrame({'date': all_dates})
dates_df['key'] = 1

In [27]:
# Full grid: each (county, place) pair × every date
full_grid = pairs.merge(dates_df, on='key').drop(columns='key')

place_day = full_grid.merge(
    place_counts,
    on=['county_fips', 'place_fips', 'date'],
    how='left'
)

In [28]:
place_day['property_crime_count'] = (
    place_day['property_crime_count'].fillna(0).astype(int)
)

In [29]:
place_day

,county_fips,place_fips,date,property_crime_count
0,01003,04660,2017-01-01,0
1,01003,04660,2017-01-02,0
2,01003,04660,2017-01-03,0
3,01003,04660,2017-01-04,0
4,01003,04660,2017-01-05,0
...,...,...,...,...
17870947,55139,55750,2024-12-27,1
17870948,55139,55750,2024-12-28,0
17870949,55139,55750,2024-12-29,1
17870950,55139,55750,2024-12-30,2


In [30]:
# Make sure date columns are datetime
county_day['date'] = pd.to_datetime(county_day['date'], errors='coerce')
place_day['date']  = pd.to_datetime(place_day['date'], errors='coerce')

# ---------------------------
# COUNTY–WEEK PANEL
# ---------------------------

In [31]:
# 1. Define week start (Monday) for each date
county_day['week_start'] = county_day['date'] - pd.to_timedelta(
    county_day['date'].dt.weekday, unit='D'
)

In [32]:
# 2. Aggregate to county–week
county_week = (
    county_day
    .groupby(['county_fips', 'week_start'], as_index=False)['property_crime_count']
    .sum()
)

In [33]:
# 3. Rename for clarity
county_week = county_week.rename(columns={'week_start': 'week'})

In [34]:
county_week

,county_fips,week,property_crime_count
0,01003,2016-12-26,0
1,01003,2017-01-02,0
2,01003,2017-01-09,0
3,01003,2017-01-16,0
4,01003,2017-01-23,0
...,...,...,...
776402,55139,2024-12-02,11
776403,55139,2024-12-09,7
776404,55139,2024-12-16,5
776405,55139,2024-12-23,4


# ---------------------------
# PLACE–WEEK PANEL
# ---------------------------

In [35]:
# 1. Define week start for each place–day
place_day['week_start'] = place_day['date'] - pd.to_timedelta(
    place_day['date'].dt.weekday, unit='D'
)

In [36]:
# 2. Aggregate to place–week
place_week = (
    place_day
    .groupby(['county_fips', 'place_fips', 'week_start'], as_index=False)['property_crime_count']
    .sum()
)

In [37]:
# 3. Rename
place_week = place_week.rename(columns={'week_start': 'week'})

In [38]:
place_week

,county_fips,place_fips,week,property_crime_count
0,01003,04660,2016-12-26,0
1,01003,04660,2017-01-02,0
2,01003,04660,2017-01-09,0
3,01003,04660,2017-01-16,0
4,01003,04660,2017-01-23,0
...,...,...,...,...
2562599,55139,55750,2024-12-02,11
2562600,55139,55750,2024-12-09,7
2562601,55139,55750,2024-12-16,5
2562602,55139,55750,2024-12-23,4


In [41]:
ccc_panel

,date,fips_code,violent_event
0,2017-01-01,11001.0,0
1,2017-01-01,27013.0,0
2,2017-01-01,27053.0,0
3,2017-01-01,44005.0,0
4,2017-01-01,47001.0,0
...,...,...,...
211350,2024-12-31,51059.0,0
211351,2024-12-31,50023.0,0
211352,2024-12-31,53033.0,0
211353,2024-12-31,53053.0,0
